In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
parquet_file_path = '../data/df_processed.parquet'
server_name = '.'
database_name = 'cyclistic_bike_database'

In [3]:
connection_string = f"mssql+pyodbc://@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

In [4]:
df = pd.read_parquet(parquet_file_path)

In [5]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week,month,hour
0,BADF67E2C5058F19,classic_bike,2025-05-11 17:22:39.471,2025-05-11 18:11:19.249,DuSable Lake Shore Dr & North Blvd,LF-005,Winthrop Ave & Lawrence Ave,TA1308000021,41.911722,-87.626804,41.968812,-87.657659,member,0 days 00:48:39.778000,Sunday,May,17
1,0210AE485D59C8C5,electric_bike,2025-05-05 08:02:09.251,2025-05-05 08:12:07.549,Damen Ave & Grand Ave,TA1308000006,Desplaines St & Jackson Blvd,15539,41.892394,-87.676885,41.878119,-87.643948,member,0 days 00:09:58.298000,Monday,May,8
2,5E68FE5B9283E4C4,classic_bike,2025-05-02 10:32:33.062,2025-05-02 10:39:07.262,LaSalle St & Illinois St,13430,Clark St & Elm St,TA1307000039,41.890762,-87.631697,41.903322,-87.632999,member,0 days 00:06:34.200000,Friday,May,10
3,13D2DCD6FB872858,classic_bike,2025-05-12 11:12:16.579,2025-05-12 11:17:25.126,Milwaukee Ave & Rockwell St,13242,Damen Ave & Cortland St,13133,41.920330,-87.693090,41.915983,-87.677335,member,0 days 00:05:08.547000,Monday,May,11
4,F04DF9EE163351DD,classic_bike,2025-05-01 10:13:36.821,2025-05-01 10:17:40.548,Halsted St & Roosevelt Rd,TA1305000017,Clinton St & Roosevelt Rd,WL-008,41.867324,-87.648625,41.867118,-87.641088,member,0 days 00:04:03.727000,Thursday,May,10


In [6]:
df.dtypes

ride_id                           str
rideable_type                     str
started_at             datetime64[us]
ended_at               datetime64[us]
start_station_name                str
start_station_id                  str
end_station_name                  str
end_station_id                    str
start_lat                     float64
start_lng                     float64
end_lat                       float64
end_lng                       float64
member_casual                     str
ride_length           timedelta64[us]
day_of_week                       str
month                             str
hour                            int32
dtype: object

In [7]:
# transform ride length column data type
df['ride_length'] = df['ride_length'].dt.total_seconds() / 60

In [8]:
df['ride_length'].describe()

count    5.691598e+06
mean     1.453000e+01
std      2.986256e+01
min     -5.479480e+01
25%      5.368567e+00
50%      9.385800e+00
75%      1.649233e+01
max      1.499968e+03
Name: ride_length, dtype: float64

In [9]:
df['ride_length'].describe()

count    5.691598e+06
mean     1.453000e+01
std      2.986256e+01
min     -5.479480e+01
25%      5.368567e+00
50%      9.385800e+00
75%      1.649233e+01
max      1.499968e+03
Name: ride_length, dtype: float64

In [10]:
df.dtypes

ride_id                          str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
ride_length                  float64
day_of_week                      str
month                            str
hour                           int32
dtype: object

_____________________________________________________________________________

*User Type Dimension*

In [11]:
user_type_dim = df[['member_casual']].drop_duplicates()

In [12]:
user_type_dim.columns = ['user_type']

In [ ]:
# user_type_dim.to_sql(
#     'dim_user_type', 
#     con=engine, 
#     if_exists='append', 
#     index=False
# )

2

Rideable Type Dimension 

In [74]:
rideable_type_dim = df[['rideable_type']].drop_duplicates()


In [75]:
rideable_type_dim.columns = ['rideable_type']

In [ ]:
# rideable_type_dim.to_sql(
#     'dim_rideable_type', 
#     con=engine, 
#     if_exists='append',
#     index=False
# )

2

Station Dimension

In [98]:
start = df[['start_station_id','start_station_name','start_lat','start_lng']]
end = df[['end_station_id','end_station_name','end_lat','end_lng']]

start.columns = ['station_id','station_name','lat','lng']
end.columns = ['station_id','station_name','lat','lng']

station_dim = pd.concat([start, end]).drop_duplicates(subset=['station_id'])

In [ ]:
# station_dim.to_sql('dim_station', engine, if_exists='append', index=False)

104

In [19]:
station_dim['station_id'].nunique(), len(station_dim)

(3247, 3248)

In [20]:
station_dim = station_dim.drop_duplicates(subset=['station_id'])

Date Dimension

In [21]:
date_dim = df[['started_at']].copy()

In [22]:
date_dim['full_date'] = date_dim['started_at'].dt.date
date_dim['day'] = date_dim['started_at'].dt.day
date_dim['month'] = date_dim['started_at'].dt.month_name()
date_dim['year'] = date_dim['started_at'].dt.year
date_dim['day_of_week'] = df['day_of_week']
date_dim['hour']=df['hour']
date_dim['day_of_month'] = date_dim['started_at'].dt.day
date_dim['is_weekend'] = date_dim['day_of_week'].isin(['Saturday','Sunday']).astype(int)


In [23]:
date_dim.head()

,started_at,full_date,day,month,year,day_of_week,hour,day_of_month,is_weekend
0,2025-05-11 17:22:39.471,2025-05-11,11,May,2025,Sunday,17,11,1
1,2025-05-05 08:02:09.251,2025-05-05,5,May,2025,Monday,8,5,0
2,2025-05-02 10:32:33.062,2025-05-02,2,May,2025,Friday,10,2,0
3,2025-05-12 11:12:16.579,2025-05-12,12,May,2025,Monday,11,12,0
4,2025-05-01 10:13:36.821,2025-05-01,1,May,2025,Thursday,10,1,0


In [24]:
date_dim.dtypes

started_at      datetime64[us]
full_date               object
day                      int32
month                      str
year                     int32
day_of_week                str
hour                     int32
day_of_month             int32
is_weekend               int64
dtype: object

In [25]:
date_dim['full_date'] = pd.to_datetime(date_dim['full_date'])

In [26]:
date_dim.dtypes

started_at      datetime64[us]
full_date        datetime64[s]
day                      int32
month                      str
year                     int32
day_of_week                str
hour                     int32
day_of_month             int32
is_weekend               int64
dtype: object

In [27]:
date_dim.duplicated().sum() 

np.int64(1097)

In [28]:
date_dim.drop_duplicates(inplace=True)

In [29]:
date_dim = date_dim[[
    'full_date', 
    'year', 
    'month', 
    'day_of_month', 
    'day_of_week', 
    'hour', 
    'is_weekend'
]]

In [30]:
date_dim.to_sql('dim_date', engine, if_exists='append', index=False)

232

Fact Table

In [31]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week,month,hour
0,BADF67E2C5058F19,classic_bike,2025-05-11 17:22:39.471,2025-05-11 18:11:19.249,DuSable Lake Shore Dr & North Blvd,LF-005,Winthrop Ave & Lawrence Ave,TA1308000021,41.911722,-87.626804,41.968812,-87.657659,member,48.662967,Sunday,May,17
1,0210AE485D59C8C5,electric_bike,2025-05-05 08:02:09.251,2025-05-05 08:12:07.549,Damen Ave & Grand Ave,TA1308000006,Desplaines St & Jackson Blvd,15539,41.892394,-87.676885,41.878119,-87.643948,member,9.971633,Monday,May,8
2,5E68FE5B9283E4C4,classic_bike,2025-05-02 10:32:33.062,2025-05-02 10:39:07.262,LaSalle St & Illinois St,13430,Clark St & Elm St,TA1307000039,41.890762,-87.631697,41.903322,-87.632999,member,6.570000,Friday,May,10
3,13D2DCD6FB872858,classic_bike,2025-05-12 11:12:16.579,2025-05-12 11:17:25.126,Milwaukee Ave & Rockwell St,13242,Damen Ave & Cortland St,13133,41.920330,-87.693090,41.915983,-87.677335,member,5.142450,Monday,May,11
4,F04DF9EE163351DD,classic_bike,2025-05-01 10:13:36.821,2025-05-01 10:17:40.548,Halsted St & Roosevelt Rd,TA1305000017,Clinton St & Roosevelt Rd,WL-008,41.867324,-87.648625,41.867118,-87.641088,member,4.062117,Thursday,May,10


Fact Table

In [32]:
fact_df = df.copy()

In [59]:
user_dim = pd.read_sql(
    "SELECT user_type_id, user_type FROM dim_user_type",
    con=engine
)

In [60]:
user_dim.duplicated().sum()

np.int64(0)

Fact TAble ((()))

In [82]:
fact_df['member_casual'] = fact_df['member_casual'].str.strip().str.lower()

In [83]:
user_dim['user_type'] = user_dim['user_type'].str.strip().str.lower()

In [84]:
user_dim.head()

,user_type_id,user_type
0,1,member
1,2,casual


In [85]:
user_lookup = user_dim.set_index('user_type')['user_type_id']

In [86]:
user_lookup.nunique()

2

In [87]:
user_dim.head()

,user_type_id,user_type
0,1,member
1,2,casual


In [88]:
fact_df['user_type_id'] = fact_df['member_casual'].map(user_lookup)

In [90]:
rideable_dim = pd.read_sql(
    "SELECT rideable_type_id, rideable_type FROM dim_rideable_type",
    con=engine
)

In [91]:
rideable_lookup = rideable_dim.set_index('rideable_type')['rideable_type_id']

In [92]:
rideable_type_dim.head()

,rideable_type
0,classic_bike
1,electric_bike


In [93]:
fact_df['rideable_type_id'] = fact_df['rideable_type'].map(rideable_lookup)

In [108]:
station_dim = pd.read_sql(
    "SELECT station_key, station_id FROM dim_station",
    con=engine
)

In [109]:
station_lookup = station_dim.set_index('station_id')['station_key']

In [110]:
fact_df['start_station_id'].duplicated().sum()
fact_df['start_station_id'].drop_duplicates(inplace=True)

In [111]:
fact_df['end_station_id'].duplicated().sum()
fact_df['end_station_id'].drop_duplicates(inplace=True)

In [112]:
fact_df['start_station_key'] = fact_df['start_station_id'].map(station_lookup)
fact_df['end_station_key'] = fact_df['end_station_id'].map(station_lookup)

In [113]:
date_dim = pd.read_sql(
    "SELECT date_id, full_date FROM dim_date",
    con=engine
)

In [114]:
date_dim.head()

,date_id,full_date
0,1,2025-05-11
1,2,2025-05-05
2,3,2025-05-02
3,4,2025-05-12
4,5,2025-05-01


In [115]:
fact_df.dtypes

ride_id                          str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
ride_length                  float64
day_of_week                      str
month                            str
hour                           int32
user_type_id                   int64
rideable_type_id               int64
start_station_key              int64
end_station_key                int64
dtype: object

In [128]:
date_dim['full_date'] = pd.to_datetime(date_dim['full_date']).dt.date
fact_df['full_date'] = pd.to_datetime(fact_df['started_at']).dt.date

In [129]:
date_dim = date_dim.drop_duplicates(subset=['full_date'])

date_lookup = date_dim.set_index('full_date')['date_id']

In [130]:
fact_df['date_id'] = fact_df['full_date'].map(date_lookup)

In [131]:
fact_table = fact_df[[
    'ride_id',
    'date_id',
    'user_type_id',
    'rideable_type_id',
    'start_station_key',
    'end_station_key',
    'started_at',
    'ended_at',
    'ride_length'
]]

In [133]:
fact_table.isna().sum()


ride_id              0
date_id              0
user_type_id         0
rideable_type_id     0
start_station_key    0
end_station_key      0
started_at           0
ended_at             0
ride_length          0
dtype: int64

In [134]:
fact_table.dtypes

ride_id                         str
date_id                       int64
user_type_id                  int64
rideable_type_id              int64
start_station_key             int64
end_station_key               int64
started_at           datetime64[us]
ended_at             datetime64[us]
ride_length                 float64
dtype: object

In [135]:
fact_table.to_sql(
    'fact_bike_trips',
    engine,
    if_exists='append',
    index=False,
    chunksize=5000
)

121966

In [137]:
fact_table.shape

(5691598, 9)

In [136]:
df.shape

(5691598, 17)